In [ ]:
import sys
import re
import pandas as pd
from sklearn.model_selection import train_test_split

# Import kamus slang dari folder src
sys.path.append('../src')
from dictionary import kamus_slang

In [ ]:
# Membaca dataset dari folder data
df = pd.read_csv('../data/tokopedia_product_reviews_2025.csv')

print(f"Jumlah data awal: {len(df)}")

# Quality Check: Hapus Missing Value & Duplikasi Teks Ulasan
df = df.dropna(subset=['review_text', 'product_category', 'sentiment_label'])
df = df.drop_duplicates(subset=['review_text'])

print(f"Jumlah data setelah Quality Check: {len(df)}")

In [ ]:
def clean_and_normalize(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Cleaning: Hapus URL, Mention (@), Hashtag (#), dan karakter khusus/angka
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[@#]\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # 2. Lowercasing
    text = text.lower()
    
    # 3. Normalisasi Slang (Pertahankan kata negasi seperti 'gak', 'tidak')
    words = text.split()
    normalized_words = [kamus_slang.get(word, word) for word in words]
    
    return " ".join(normalized_words).strip()

# Jalankan fungsi pada dataset
df['clean_review'] = df['review_text'].apply(clean_and_normalize)

# Tampilkan 5 baris pertama hasil pembersihan
df[['review_text', 'clean_review']].head()

In [ ]:
# Gabungkan Aspek (Kategori Produk) dan Teks Ulasan dengan token [SEP]
df['absa_input'] = df['product_category'] + " [SEP] " + df['clean_review']

# Split Data (80% Train, 10% Validation, 10% Test)
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['sentiment_label']
)
val_df, test_df = train_test_split(
    test_df, test_size=0.5, random_state=42, stratify=test_df['sentiment_label']
)

# Simpan hasil ke folder data/
train_df.to_csv('../data/train.csv', index=False)
val_df.to_csv('../data/val.csv', index=False)
test_df.to_csv('../data/test.csv', index=False)

print("--- Preprocessing Selesai ---")
print(f"Data Train      : {len(train_df)} baris")
print(f"Data Validation : {len(val_df)} baris")
print(f"Data Test       : {len(test_df)} baris")